# RF-DETR vs YOLOv12 vs YOLO26 — unified RF100 benchmark

Each model is fine-tuned **separately** on three RF100 datasets (`cable-damage`,
`bone-fracture-7fylg`, `soda-bottles`). The per-model `finetune_rf100.py` scripts write a
standardized JSON into `comparison_results/`; this notebook reads all three and compares them.

Produce the inputs first (each from its own directory + venv):

```bash
cd rf-detr  && .venv/bin/python finetune_rf100.py --epochs 10 --skip-existing
cd yolov12  && .venv/bin/python finetune_rf100.py --epochs 10
cd yolov26  && .venv/bin/python finetune_rf100.py --epochs 10
```

This notebook only needs `matplotlib` + stdlib, so any of the three venvs can run it.
It skips gracefully any model whose JSON is not present yet.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from urllib3.util.proxy import connection_requires_http_tunnel

RESULTS_DIR = Path("comparison_results")
MODEL_FILES = {
    "rfdetr_nano": "rfdetr_nano_results.json",
    "rfdetr_large": "rfdetr_large_results.json",
    "rfdetr_base": "rfdetr_base_results.json",
    "yolov12n": "yolov12n_results.json",
    "yolo26n": "yolo26n_results.json",
    "yolov12m": "yolov12m_results.json",
    "yolo26m": "yolo26m_results.json",
}

results = {}
for model, fname in MODEL_FILES.items():
    path = RESULTS_DIR / fname
    if path.exists():
        results[model] = json.loads(path.read_text())
        print(f"loaded {model}: {list(results[model]['datasets'])}")
    else:
        print(f"MISSING {model}: {path} (run its finetune_rf100.py first)")

# Union of datasets present across whatever models we loaded.
datasets = sorted({ds for r in results.values() for ds in r.get("datasets", {})})
print("\ndatasets:", datasets)

# Optional per-model metadata (present after running the param-count cells / COCO val cells).
def _load(name):
    fp = RESULTS_DIR / name
    return json.loads(fp.read_text()) if fp.exists() else {}

model_params = _load("model_params.json")     # {model: {"num_params": int}}
coco_baseline = _load("coco_baseline.json")   # {model: {"mAP50":.., "mAP50_95":..}}

def fmt(x):
    return "  n/a" if x is None else f"{x:.4f}"


## Model card — size vs. COCO reference

Per-model **parameter count** and the one-shot **COCO val2017** mAP of the pretrained weights (no fine-tune). Populate these by running the param-count cell and the COCO-val cell in each `test.ipynb` (or `prepare_coco_val.py` + those cells).

In [ ]:
hdr = f"{'model':<14}{'params (M)':>12}{'COCO mAP50':>12}{'COCO 50-95':>12}{'COCO FPS':>12}"
print(hdr)
print("-" * len(hdr))
for model in MODEL_FILES:
    n = model_params.get(model, {}).get("num_params")
    cc = coco_baseline.get(model, {})
    pm = f"{n / 1e6:.2f}" if n else "  n/a"
    fps = cc.get("fps")
    fps_s = f"{fps:.1f}" if fps else "  n/a"
    print(f"{model:<14}{pm:>12}{fmt(cc.get('mAP50')):>12}{fmt(cc.get('mAP50_95')):>12}{fps_s:>12}")


## Summary table — zero-shot baseline vs final test

Baseline is the pretrained (COCO) model evaluated before fine-tuning. It is expected to be
≈ 0 because COCO class ids do not match the relabeled RF100 class ids — it is the honest
"before" reference, not a meaningful score.

In [ ]:
header = f"{'dataset':<22}{'model':<14}{'base mAP50':>12}{'base 50-95':>12}{'test mAP50':>12}{'test 50-95':>12}"
print(header)
print("-" * len(header))
for ds in datasets:
    for model in MODEL_FILES:
        entry = results.get(model, {}).get("datasets", {}).get(ds)
        if not entry:
            continue
        b = entry.get("baseline_val", {})
        t = entry.get("final_test", {})
        print(
            f"{ds:<22}{model:<14}"
            f"{fmt(b.get('mAP50')):>12}{fmt(b.get('mAP50_95')):>12}"
            f"{fmt(t.get('mAP50')):>12}{fmt(t.get('mAP50_95')):>12}"
        )
    print()

## Validation curves during fine-tuning

One figure per dataset; each line is a model's validation **mAP50-95** per epoch. Switch
`METRIC` to `"mAP50"` to compare that instead.

In [ ]:
METRIC = "mAP50_95"  # or "mAP50"

for ds in datasets:
    plt.figure(figsize=(8, 5))
    plotted = 0
    for model in MODEL_FILES:
        entry = results.get(model, {}).get("datasets", {}).get(ds)
        if not entry or not entry.get("val_curve"):
            continue
        curve = entry["val_curve"]
        xs = [c["epoch"] for c in curve if c.get(METRIC) is not None]
        ys = [c[METRIC] for c in curve if c.get(METRIC) is not None]
        if xs:
            plt.plot(xs, ys, marker="o", linewidth=1.8, label=model)
            plotted += 1
    if not plotted:
        plt.close()
        print(f"no curve data for {ds}")
        continue
    plt.title(f"{ds}: validation {METRIC} vs epoch")
    plt.xlabel("epoch")
    plt.ylabel(f"val {METRIC}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Final test performance — grouped bar chart

Final `test`-split mAP50-95 after fine-tuning, grouped by dataset with one bar per model.

In [ ]:
TEST_METRIC = "mAP50_95"  # or "mAP50"
models_present = [m for m in MODEL_FILES if m in results]

x = range(len(datasets))
width = 0.8 / max(1, len(models_present))
plt.figure(figsize=(10, 6))
for i, model in enumerate(models_present):
    vals = []
    for ds in datasets:
        entry = results[model].get("datasets", {}).get(ds, {})
        vals.append((entry.get("final_test") or {}).get(TEST_METRIC) or 0.0)
    offsets = [xi + i * width for xi in x]
    bars = plt.bar(offsets, vals, width=width, label=model)
    for rect, v in zip(bars, vals):
        plt.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.2f}",
                 ha="center", va="bottom", fontsize=8)

plt.xticks([xi + width * (len(models_present) - 1) / 2 for xi in x], datasets, rotation=15)
plt.ylabel(f"test {TEST_METRIC}")
plt.title(f"Final test {TEST_METRIC} by model and dataset")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Predicted bounding boxes on test images

Each model draws boxes on the **same** first-K test images per dataset (so the grid lines
up), rendered to `comparison_results/pred_viz/<model>/<dataset>/`. Generate them once per
model, from each model's own directory + venv (inference only — no retraining):

```bash
cd rf-detr  && .venv/bin/python render_predictions.py --samples 6
cd yolov12  && .venv/bin/python render_predictions.py --samples 6
cd yolov26  && .venv/bin/python render_predictions.py --samples 6
```

The grid below has one figure per dataset. The **first row is the ground truth** (drawn
here directly from the RF100 `test/labels/*.txt` YOLO annotations, colored per class with
a legend), followed by one **row per model** and one **column per test image**. Ground-truth
boxes are read live from `RF100_ROOT`, so that row renders even before any predictions exist.
Tune `MAX_IMAGES` to show more/fewer columns.

In [ ]:
import matplotlib.patches as patches

PRED_VIZ = RESULTS_DIR / "pred_viz"
RF100_ROOT = Path("/home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MODEL_ROWS = ["rfdetr_nano", "rfdetr_base", "rfdetr_large", "yolov12n", "yolov12m", "yolo26n", "yolo26m"]  # model row order (ground truth is always row 0)
MAX_IMAGES = 6

# One color per class id, reused for every ground-truth box in a dataset so the
# GT row is internally consistent (class k -> same color across all images).
GT_PALETTE = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
              "#46f0f0", "#f032e6", "#bcf60c", "#fabebe", "#008080"]

plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600

_class_name_cache = {}

def load_class_names(ds):
    """Parse the `names:` block-list from a dataset's data.yaml (stdlib only)."""
    if ds in _class_name_cache:
        return _class_name_cache[ds]
    names, yml = [], RF100_ROOT / ds / "data.yaml"
    if yml.exists():
        in_names = False
        for line in yml.read_text().splitlines():
            if line.startswith("names:"):
                rest = line.split(":", 1)[1].strip()
                if rest.startswith("["):  # inline list form: names: [a, b]
                    names = [s.strip().strip("'\"") for s in rest.strip("[]").split(",") if s.strip()]
                    break
                in_names = True
                continue
            if in_names:
                s = line.strip()
                if s.startswith("-"):
                    names.append(s[1:].strip().strip("'\""))
                elif s:  # first non-list line ends the block
                    break
    _class_name_cache[ds] = names
    return names

def draw_ground_truth(ax, ds, stem):
    """Draw YOLO-format GT boxes (test/labels/<stem>.txt) on the original test image."""
    img_dir = RF100_ROOT / ds / "test" / "images"
    img_p = next((p for p in sorted(img_dir.glob(stem + ".*")) if p.suffix.lower() in IMAGE_EXTS), None)
    if img_p is None:
        ax.text(0.5, 0.5, "(no image)", ha="center", va="center", fontsize=8)
        return
    img = mpimg.imread(img_p)
    h, w = img.shape[:2]
    ax.imshow(img)
    names = load_class_names(ds)
    lbl = RF100_ROOT / ds / "test" / "labels" / (stem + ".txt")
    if not lbl.exists():
        return
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(float(parts[0]))
        cx, cy, bw, bh = map(float, parts[1:5])
        x, y = (cx - bw / 2) * w, (cy - bh / 2) * h
        color = GT_PALETTE[cid % len(GT_PALETTE)]
        ax.add_patch(patches.Rectangle((x, y), bw * w, bh * h,
                                       linewidth=1.5, edgecolor=color, facecolor="none"))
        name = names[cid] if cid < len(names) else str(cid)
        ax.text(x, y - 2, name, color="white", fontsize=6, va="bottom",
                bbox=dict(facecolor=color, edgecolor="none", pad=0.5, alpha=0.85))

def show_dataset_predictions(ds):
    per_model = {}
    for m in MODEL_ROWS:
        d = PRED_VIZ / m / ds
        per_model[m] = {p.stem: p for p in sorted(d.glob("*.png"))} if d.is_dir() else {}
    models_here = [m for m in MODEL_ROWS if per_model[m]]

    # Choose image stems. Prefer stems present for every model so rows line up;
    # if no predictions exist yet, fall back to the first-K dataset test images
    # (same sort order the render scripts use) so the GT row still renders.
    if models_here:
        all_stems = sorted(set().union(*[set(per_model[m]) for m in models_here]))
        common = [s for s in all_stems if all(s in per_model[m] for m in models_here)]
        chosen = (common or all_stems)[:MAX_IMAGES]
    else:
        img_dir = RF100_ROOT / ds / "test" / "images"
        chosen = ([p.stem for p in sorted(img_dir.iterdir()) if p.suffix.lower() in IMAGE_EXTS][:MAX_IMAGES]
                  if img_dir.is_dir() else [])
    if not chosen:
        print(f"no images for {ds} - run render_predictions.py in each model dir")
        return

    row_labels = ["ground_truth"] + models_here
    rows, cols = len(row_labels), len(chosen)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows), squeeze=False)

    # Row 0: ground truth.
    for c, stem in enumerate(chosen):
        ax = axes[0][c]
        ax.set_xticks([]); ax.set_yticks([])
        draw_ground_truth(ax, ds, stem)
        ax.set_title(stem[:16], fontsize=7)
        if c == 0:
            ax.set_ylabel("ground_truth", fontsize=11)

    # Rows 1..N: model predictions (pre-rendered PNGs).
    for r, m in enumerate(models_here, start=1):
        for c, stem in enumerate(chosen):
            ax = axes[r][c]
            ax.set_xticks([]); ax.set_yticks([])
            p = per_model[m].get(stem)
            if p is not None:
                ax.imshow(mpimg.imread(p))
            else:
                ax.text(0.5, 0.5, "(no image)", ha="center", va="center", fontsize=8)
            if c == 0:
                ax.set_ylabel(m, fontsize=11)

    # Class -> color legend for the ground-truth row.
    names = load_class_names(ds)
    handles = [patches.Patch(color=GT_PALETTE[i % len(GT_PALETTE)], label=n) for i, n in enumerate(names)]
    if handles:
        fig.legend(handles=handles, loc="upper right", fontsize=8, ncol=min(len(handles), 4),
                   title="GT classes", title_fontsize=8)

    fig.suptitle(f"{ds}: ground truth + test predictions  (rows = GT then models, cols = images)", fontsize=13)
    plt.tight_layout()
    plt.show()

for ds in datasets:
    show_dataset_predictions(ds)

## Accuracy vs. speed trade-off

Batch=1 inference speed (measured with warmup + CUDA sync, the same way for every model) against mAP50-95. **Left:** COCO val2017 with pretrained weights (one point per model). **Right:** RF100 fine-tuned models on each dataset's test split (one point per model x dataset). Up-and-to-the-right is better. Needs the speed fields populated by the COCO-val cells and the `finetune_rf100.py` runs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: COCO val2017, one point per model.
ax = axes[0]
for model in MODEL_FILES:
    cc = coco_baseline.get(model, {})
    if cc.get("fps") and cc.get("mAP50_95") is not None:
        ax.scatter(cc["fps"], cc["mAP50_95"], s=90)
        ax.annotate(model, (cc["fps"], cc["mAP50_95"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("FPS (batch=1)"); ax.set_ylabel("COCO val2017 mAP50-95")
ax.set_title("COCO (pretrained): accuracy vs speed"); ax.grid(True, alpha=0.3)

# Right: RF100 fine-tuned test split, one point per (model, dataset).
ax = axes[1]
for model in MODEL_FILES:
    xs, ys = [], []
    for ds in datasets:
        print(f"Model: {model}, Dataset: {ds}")
        ft = (results.get(model, {}).get("datasets", {}).get(ds, {}) or {}).get("final_test", {}) or {}
        if ft.get("fps") and ft.get("mAP50_95") is not None:
            xs.append(ft["fps"]); ys.append(ft["mAP50_95"])
    if xs:
        ax.scatter(xs, ys, s=80, label=model)
ax.set_xlabel("FPS (batch=1)"); ax.set_ylabel("RF100 test mAP50-95")
ax.set_title("RF100 (fine-tuned): accuracy vs speed"); ax.grid(True, alpha=0.3)
if ax.get_legend_handles_labels()[0]:
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: COCO val2017, one point per model.
ax = axes[0]
for model in MODEL_FILES:
    cc = coco_baseline.get(model, {})
    if cc.get("fps") and cc.get("mAP50_95") is not None:
        ax.scatter(cc["fps"], cc["mAP50_95"], s=90)
        ax.annotate(
            model,
            (cc["fps"], cc["mAP50_95"]),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=9,
        )
ax.set_xlabel("FPS (batch=1)")
ax.set_ylabel("COCO val2017 mAP50-95")
ax.set_title("COCO (pretrained): accuracy vs speed")
ax.grid(True, alpha=0.3)

# Right: RF100 fine-tuned test split, one point per (model, dataset).
ax = axes[1]
for model in MODEL_FILES:
    if model == "rfdetr_base":
       continue
    xs, ys = [], []
    for ds in datasets:
        ft = (results.get(model, {}).get("datasets", {}).get(ds, {}) or {}).get("final_test", {}) or {}
        if ft.get("fps") and ft.get("mAP50_95") is not None:
            fps = ft["fps"]
            map50_95 = ft["mAP50_95"]
            xs.append(fps)
            ys.append(map50_95)
            ax.annotate(
                f"{ds}",
                (fps, map50_95),
                textcoords="offset points",
                xytext=(6, 4),
                fontsize=7,
                alpha=0.85,
            )
    if xs:
        ax.scatter(xs, ys, s=80, label=model)

ax.set_xlabel("FPS (batch=1)")
ax.set_ylabel("RF100 test mAP50-95")
ax.set_title("RF100 (fine-tuned): accuracy vs speed")
ax.grid(True, alpha=0.3)
if ax.get_legend_handles_labels()[0]:
    ax.legend()
plt.tight_layout()
plt.show()